# SageMaker CI/CD Pipeline for Sound Emotion Classification

This notebook provides a repeatable ML workflow:
1. Preprocess data (train/test split)
2. Train XGBoost model
3. Evaluate model performance
4. Conditionally register model if accuracy >= threshold

**Dataset:** Sound emotion classification (7 classes)  
**Location:** `s3://sagemaker-us-east-1-418418308994/models/benchmarks/baseline_normalized.csv`

---

**Run this pipeline whenever you want to retrain/redeploy the model.**  
After deployment, use your monitoring setup notebook to configure monitoring infrastructure.

**Attribution:** This was created from a lab in AAI-540 at University of San Diego, as well as with the assistance of Claude Code accessed February 2026. 

## 1. Setup, Config & Pipeline Parameters

In [19]:
import boto3
import sagemaker
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.steps import ProcessingStep, TrainingStep
from sagemaker.workflow.step_collections import RegisterModel
from sagemaker.workflow.parameters import ParameterString, ParameterFloat
from sagemaker.workflow.properties import PropertyFile
from sagemaker.workflow.conditions import ConditionGreaterThanOrEqualTo
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.workflow.functions import JsonGet, Join
from sagemaker.workflow.pipeline_context import PipelineSession
from sagemaker.processing import ProcessingInput, ProcessingOutput, ScriptProcessor
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.inputs import TrainingInput
from sagemaker.xgboost import XGBoost
from sagemaker.model_metrics import MetricsSource, ModelMetrics
from pathlib import Path
import json
import datetime
import time
from datetime import timezone
from urllib.parse import urlparse

print("✅ Imports complete")

✅ Imports complete


In [2]:
#AWS Core Configuration
region = boto3.Session().region_name
sagemaker_session = sagemaker.Session()
pipeline_session = PipelineSession()
bucket = sagemaker_session.default_bucket()
role = sagemaker.get_execution_role()
prefix = "models/benchmarks"

sm_client = boto3.client('sagemaker', region_name=region)
s3_client = boto3.client('s3', region_name=region)

print(f"\nRegion: {region}")
print(f"Bucket: {bucket}")
print(f"Prefix: {prefix}")
print(f"Role: {role}")


Region: us-east-1
Bucket: sagemaker-us-east-1-513691803389
Prefix: models/benchmarks
Role: arn:aws:iam::513691803389:role/LabRole


### Pipeline Parameters

These parameters can be changed when starting the pipeline without redefining it.

In [3]:
# Input data location
input_data = ParameterString(
    name="InputData",
    default_value=f"s3://{bucket}/{prefix}/baseline_normalized.csv"
)

# Instance types
processing_instance_type = ParameterString(
    name="ProcessingInstanceType",
    default_value="ml.m5.xlarge"
)

training_instance_type = ParameterString(
    name="TrainingInstanceType",
    default_value="ml.m5.xlarge"
)

# Model approval
model_approval_status = ParameterString(
    name="ModelApprovalStatus",
    default_value="PendingManualApproval"
)

# Accuracy threshold for registration
accuracy_threshold = ParameterFloat(
    name="AccuracyThreshold",
    default_value=0.6 #Model achieved accuracy of .62 previously in repository
)

print("✅ Parameters defined")

✅ Parameters defined


## 2. Create Processing Scripts

These scripts will be packaged and uploaded by SageMaker.

In [4]:
# 2.1 Preprocessing Script
preprocessing_script = """
import pandas as pd
from sklearn.model_selection import train_test_split
import argparse
import os

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--test-size", type=float, default=0.2)
    args, _ = parser.parse_known_args()
    
    # Load data
    input_path = "/opt/ml/processing/input/baseline_normalized.csv"
    df = pd.read_csv(input_path)
    
    print(f"Loaded dataset: {df.shape}")
    print(f"Target distribution:\\n{df['target'].value_counts()}")
    
    # Split features and target
    X = df.drop('target', axis=1)
    y = df['target']
    
    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=args.test_size, random_state=42, stratify=y
    )
    
    # Combine for XGBoost format (target first)
    train_data = pd.concat([y_train.reset_index(drop=True), 
                            X_train.reset_index(drop=True)], axis=1)
    test_data = pd.concat([y_test.reset_index(drop=True), 
                           X_test.reset_index(drop=True)], axis=1)
    
    print(f"Train size: {len(train_data)}, Test size: {len(test_data)}")
    
    # Save
    os.makedirs("/opt/ml/processing/train", exist_ok=True)
    os.makedirs("/opt/ml/processing/test", exist_ok=True)
    
    train_data.to_csv("/opt/ml/processing/train/train.csv", index=False, header=False)
    test_data.to_csv("/opt/ml/processing/test/test.csv", index=False, header=False)
    
    print("✅ Preprocessing complete")
"""

Path("preprocessing.py").write_text(preprocessing_script)
print("✅ Created preprocessing.py")

✅ Created preprocessing.py


In [5]:
#2.2 Training script
train_script = """
import argparse
import os
import pandas as pd
import xgboost as xgb
import pickle

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    
    # Hyperparameters
    parser.add_argument("--objective", type=str, default="multi:softmax")
    parser.add_argument("--num_class", type=int, default=7)
    parser.add_argument("--num_round", type=int, default=100)
    parser.add_argument("--max_depth", type=int, default=5)
    parser.add_argument("--eta", type=float, default=0.2)
    parser.add_argument("--gamma", type=float, default=4)
    parser.add_argument("--min_child_weight", type=int, default=6)
    parser.add_argument("--subsample", type=float, default=0.8)
    parser.add_argument("--verbosity", type=int, default=1)
    
    args, _ = parser.parse_known_args()
    
    # Load training data
    train_path = os.path.join("/opt/ml/input/data/train", "train.csv")
    train_data = pd.read_csv(train_path, header=None)
    
    print(f"Training data shape: {train_data.shape}")
    
    # Prepare data for XGBoost
    y_train = train_data.iloc[:, 0]
    X_train = train_data.iloc[:, 1:]
    
    dtrain = xgb.DMatrix(X_train, label=y_train)
    
    # Set parameters
    params = {
        'objective': args.objective,
        'num_class': args.num_class,
        'max_depth': args.max_depth,
        'eta': args.eta,
        'gamma': args.gamma,
        'min_child_weight': args.min_child_weight,
        'subsample': args.subsample,
        'verbosity': args.verbosity
    }
    
    # Train model
    print(f"Training with params: {params}")
    model = xgb.train(params, dtrain, num_boost_round=args.num_round)
    
    # Save model
    model_dir = "/opt/ml/model"
    os.makedirs(model_dir, exist_ok=True)
    model_path = os.path.join(model_dir, "xgboost-model")
    
    with open(model_path, "wb") as f:
        pickle.dump(model, f)
    
    print(f"✅ Model saved to {model_path}")
"""

Path("train.py").write_text(train_script)
print("✅ Created train.py")

✅ Created train.py


In [6]:
#2.3 Evaluation Script
evaluation_script = """
import json
import os
import pickle
import tarfile
import pandas as pd
import xgboost as xgb
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

if __name__ == "__main__":
    # Extract model
    model_path = "/opt/ml/processing/model/model.tar.gz"
    with tarfile.open(model_path) as tar:
        tar.extractall(path="/opt/ml/processing/model")
    
    # Load model
    with open("/opt/ml/processing/model/xgboost-model", "rb") as f:
        model = pickle.load(f)
    
    # Load test data
    test_path = "/opt/ml/processing/test/test.csv"
    test_data = pd.read_csv(test_path, header=None)
    
    y_test = test_data.iloc[:, 0]
    X_test = test_data.iloc[:, 1:]
    
    print(f"Test data shape: {X_test.shape}")
    
    # Predict
    dtest = xgb.DMatrix(X_test)
    predictions = model.predict(dtest)
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test, predictions, average='weighted', zero_division=0
    )
    
    print(f"\\nModel Performance:")
    print(f"  Accuracy:  {accuracy:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  F1 Score:  {f1:.4f}")
    
    # Create evaluation report
    report = {
        "multiclass_classification_metrics": {
            "accuracy": {"value": accuracy},
            "precision": {"value": precision},
            "recall": {"value": recall},
            "f1": {"value": f1}
        }
    }
    
    # Save report
    output_dir = "/opt/ml/processing/evaluation"
    os.makedirs(output_dir, exist_ok=True)
    
    with open(f"{output_dir}/evaluation.json", "w") as f:
        json.dump(report, f)
    
    print("✅ Evaluation complete")
"""

Path("evaluation.py").write_text(evaluation_script)
print("✅ Created evaluation.py")

✅ Created evaluation.py


## 3. Data Preprocessing

Split the dataset into training and test sets.

In [7]:
#3.1 Preprocessing Definition

sklearn_processor = SKLearnProcessor(
    framework_version="1.2-1",
    instance_type=processing_instance_type,
    instance_count=1,
    base_job_name="sound-emotion-preprocess",
    role=role,
    sagemaker_session=pipeline_session
)

processor_args = sklearn_processor.run(
    code="preprocessing.py",
    inputs=[
        ProcessingInput(
            source=input_data,
            destination="/opt/ml/processing/input"
        )
    ],
    outputs=[
        ProcessingOutput(
            output_name="train_data",
            source="/opt/ml/processing/train",
            destination=f"s3://{bucket}/{prefix}/pipeline/train"
        ),
        ProcessingOutput(
            output_name="test_data",
            source="/opt/ml/processing/test",
            destination=f"s3://{bucket}/{prefix}/pipeline/test"
        )
    ]
)

step_process = ProcessingStep(
    name="PreprocessSoundData",
    step_args=processor_args
)

print("✅ Preprocessing step defined")

INFO:sagemaker.image_uris:Defaulting to only available Python version: py3


✅ Preprocessing step defined


/opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline_context.py:332: UserWarning: Running within a PipelineSession, there will be No Wait, No Logs, and No Job being started.
  warnings.warn(


## 4. Model Training

Train an XGBoost model for multiclass classification (7 emotion classes).

In [8]:
# XGBoost hyperparameters
hyperparameters = {
    "objective": "multi:softmax",
    "num_class": "7",
    "num_round": "100",
    "max_depth": "5",
    "eta": "0.2",
    "gamma": "4",
    "min_child_weight": "6",
    "subsample": "0.8",
    "verbosity": "1"
}

xgboost_estimator = XGBoost(
    entry_point="train.py",
    framework_version="1.7-1",
    hyperparameters=hyperparameters,
    role=role,
    instance_count=1,
    instance_type=training_instance_type,
    output_path=f"s3://{bucket}/{prefix}/pipeline/models",
    base_job_name="sound-emotion-train",
    sagemaker_session=pipeline_session
)

training_args = xgboost_estimator.fit(
    inputs={
        "train": TrainingInput(
            s3_data=step_process.properties.ProcessingOutputConfig.Outputs[
                "train_data"
            ].S3Output.S3Uri,
            content_type="text/csv"
        )
    }
)

step_train = TrainingStep(
    name="TrainXGBoostModel",
    step_args=training_args
)

print("✅ Training step defined")

INFO:sagemaker.image_uris:Ignoring unnecessary Python version: py3.
INFO:sagemaker.image_uris:Ignoring unnecessary instance type: ml.m5.xlarge.
INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.


✅ Training step defined


## 5. Model Evaluation

Evaluate the trained model on the test set.

In [9]:
script_processor = ScriptProcessor(
    image_uri=xgboost_estimator.training_image_uri(),
    command=["python3"],
    instance_type=processing_instance_type,
    instance_count=1,
    base_job_name="sound-emotion-eval",
    role=role,
    sagemaker_session=pipeline_session
)

evaluation_args = script_processor.run(
    code="evaluation.py",
    inputs=[
        ProcessingInput(
            source=step_train.properties.ModelArtifacts.S3ModelArtifacts,
            destination="/opt/ml/processing/model"
        ),
        ProcessingInput(
            source=step_process.properties.ProcessingOutputConfig.Outputs[
                "test_data"
            ].S3Output.S3Uri,
            destination="/opt/ml/processing/test"
        )
    ],
    outputs=[
        ProcessingOutput(
            output_name="evaluation",
            source="/opt/ml/processing/evaluation",
            destination=f"s3://{bucket}/{prefix}/pipeline/evaluation"
        )
    ]
)

evaluation_report = PropertyFile(
    name="EvaluationReport",
    output_name="evaluation",
    path="evaluation.json"
)

step_eval = ProcessingStep(
    name="EvaluateModel",
    step_args=evaluation_args,
    property_files=[evaluation_report]
)

print("✅ Evaluation step defined")

✅ Evaluation step defined


## 6. Conditional Model Registration

Register the model only if accuracy >= threshold.

In [10]:
# Model metrics for registration
model_metrics = ModelMetrics(
    model_statistics=MetricsSource(
        s3_uri=Join(
            on="/",
            values=[
                step_eval.properties.ProcessingOutputConfig.Outputs['evaluation'].S3Output.S3Uri,
                "evaluation.json"
            ]
        ),
        content_type="application/json"
    )
)

# Register model step
step_register = RegisterModel(
    name="RegisterSoundEmotionModel",
    estimator=xgboost_estimator,
    model_data=step_train.properties.ModelArtifacts.S3ModelArtifacts,
    content_types=["text/csv"],
    response_types=["text/csv"],
    inference_instances=["ml.t2.medium", "ml.m5.large"],
    transform_instances=["ml.m5.large"],
    model_package_group_name="SoundEmotionModelGroup",
    approval_status=model_approval_status,
    model_metrics=model_metrics
)

# Condition: Only register if accuracy >= threshold
cond_gte = ConditionGreaterThanOrEqualTo(
    left=JsonGet(
        step_name=step_eval.name,
        property_file=evaluation_report,
        json_path="multiclass_classification_metrics.accuracy.value"
    ),
    right=accuracy_threshold
)

step_cond = ConditionStep(
    name="CheckAccuracyThreshold",
    conditions=[cond_gte],
    if_steps=step_register.steps,
    else_steps=[]
)

print("✅ Conditional registration defined")

✅ Conditional registration defined


## 7. Create & Execute Pipeline

In [11]:
#7.1 Pipeline Creation
pipeline_name = "SoundEmotionClassificationPipeline"

pipeline = Pipeline(
    name=pipeline_name,
    parameters=[
        input_data,
        processing_instance_type,
        training_instance_type,
        model_approval_status,
        accuracy_threshold
    ],
    steps=[step_process, step_train, step_eval, step_cond],
    sagemaker_session=pipeline_session
)

# Create or update pipeline
pipeline.upsert(role_arn=role)
print(f"✅ Pipeline created: {pipeline_name}")

✅ Pipeline created: SoundEmotionClassificationPipeline


In [12]:
#7.2 Pipeline Execution
execution = pipeline.start()
print(f"✅ Pipeline execution started!")
print(f"   Execution ARN: {execution.arn}")

✅ Pipeline execution started!
   Execution ARN: arn:aws:sagemaker:us-east-1:513691803389:pipeline/SoundEmotionClassificationPipeline/execution/hhkamta25ac9


In [13]:
#7.3 Wait for completion
print("Waiting for pipeline to complete...")
print("(This may take 10-15 minutes)\n")
execution.wait()

# Get execution details
execution_desc = execution.describe()
status = execution_desc['PipelineExecutionStatus']

print(f"\n✅ Pipeline execution complete!")
print(f"   Status: {status}")

Waiting for pipeline to complete...
(This may take 10-15 minutes)


✅ Pipeline execution complete!
   Status: Succeeded


## 8. View Results

In [15]:
#8.1 Confirm pipeline ran, and check if threshold was met
steps = execution.list_steps()
print("=" * 80)
print("PIPELINE STEPS - DETAILED STATUS")
print("=" * 80)

for step in steps:
    step_name = step['StepName']
    step_status = step['StepStatus']
    
    # Icon based on status
    icon = "✅" if step_status == "Succeeded" else "❌" if step_status == "Failed" else "⏳"
    
    print(f"\n{icon} Step: {step_name}")
    print(f"   Status: {step_status}")
    
    # Check for condition step
    if "CheckAccuracyThreshold" in step_name:
        try:
            outcome = step['Metadata']['Condition']['Outcome']
            
            # outcome == True means condition PASSED
            condition_passed = (outcome == True or outcome == "true" or outcome == "True")
            
            print(f"\n   🎯 CONDITION EVALUATION:")
            print(f"   Outcome: {outcome}")
            
            # FIXED LOGIC
            if condition_passed:
                print(f"   Result: PASSED ✅ - Model WILL be registered")
            else:
                print(f"   Result: FAILED ❌ - Model will NOT be registered")
            
            # Get threshold value
            try:
                exec_desc = execution.describe()
                params_list = exec_desc.get("PipelineParameters", [])
                params = {p["Name"]: p["Value"] for p in params_list}
                
                if "AccuracyThreshold" in params:
                    threshold_value = float(params["AccuracyThreshold"])
                else:
                    threshold_value = 0.6  #HARD CODED
                
                print(f"   📏 Threshold: {threshold_value:.4f} ({threshold_value * 100:.1f}%)")
                
            except Exception as e:
                print(f"   ⚠️  Could not get threshold: {e}")
                threshold_value = 0.6 #HARD CODED 
            
            # Get actual accuracy value
            try:
                eval_step = next((s for s in steps if s['StepName'] == 'EvaluateModel'), None)
                if eval_step and eval_step['StepStatus'] == 'Succeeded':
                    job_arn = eval_step['Metadata']['ProcessingJob']['Arn']
                    job_name = job_arn.split('/')[-1]
                    job_details = sm_client.describe_processing_job(ProcessingJobName=job_name)
                    
                    s3_uri = job_details['ProcessingOutputConfig']['Outputs'][0]['S3Output']['S3Uri']
                    eval_json_uri = f"{s3_uri}/evaluation.json"
                    
                    bucket_name = eval_json_uri.split('/')[2]
                    key = '/'.join(eval_json_uri.split('/')[3:])
                    s3_client.download_file(bucket_name, key, '/tmp/evaluation.json')
                    
                    with open('/tmp/evaluation.json', 'r') as f:
                        eval_data = json.load(f)
                    
                    accuracy = eval_data['multiclass_classification_metrics']['accuracy']['value']
                    print(f"   📊 Model Accuracy: {accuracy:.4f} ({accuracy * 100:.2f}%)")
                    
                    # FIXED: Show correct comparison
                    if condition_passed:
                        print(f"   ✓ {accuracy * 100:.2f}% ≥ {threshold_value * 100:.1f}% → Model Registered ✅")
                    else:
                        print(f"   ✗ {accuracy * 100:.2f}% < {threshold_value * 100:.1f}% → Model NOT Registered ❌")
                    
            except Exception as e:
                print(f"   ⚠️  Could not retrieve accuracy: {e}")
                
        except KeyError:
            print("   No condition metadata available")

print("\n" + "=" * 80)
print("SUMMARY")
print("=" * 80)

# Check if RegisterModel step ran
register_step = next((s for s in steps if 'Register' in s['StepName']), None)

if register_step:
    print(f"✅ RegisterSoundEmotionModel step: {register_step['StepStatus']}")
    print(f"   Model WAS registered to the registry")
    print(f"   Reason: Accuracy ({accuracy * 100:.2f}%) met threshold ({threshold_value * 100:.1f}%)")
else:
    print(f"⏭️  RegisterSoundEmotionModel step: SKIPPED")
    print(f"   Model was NOT registered (accuracy below threshold)")

print("=" * 80)

PIPELINE STEPS - DETAILED STATUS

✅ Step: RegisterSoundEmotionModel-RegisterModel
   Status: Succeeded

✅ Step: CheckAccuracyThreshold
   Status: Succeeded

   🎯 CONDITION EVALUATION:
   Outcome: True
   Result: PASSED ✅ - Model WILL be registered
   📏 Threshold: 0.6000 (60.0%)
   📊 Model Accuracy: 0.6081 (60.81%)
   ✓ 60.81% ≥ 60.0% → Model Registered ✅

✅ Step: EvaluateModel
   Status: Succeeded

✅ Step: TrainXGBoostModel
   Status: Succeeded

✅ Step: PreprocessSoundData
   Status: Succeeded

SUMMARY
✅ RegisterSoundEmotionModel step: Succeeded
   Model WAS registered to the registry
   Reason: Accuracy (60.81%) met threshold (60.0%)


## 9. Show The Model Not Passing, Adjust Threshold Higher

In [16]:
#9.1 Rerun with higher threshold
execution = pipeline.start(parameters={
    "AccuracyThreshold": 0.95  # 95% - will fail the condition
})

print(f"✅ Pipeline execution started!")
print(f"   Execution ARN: {execution.arn}")
print(f"   Accuracy Threshold: 0.95 (will likely FAIL)")

# Wait for completion (this may take 10-15 minutes)
print("Waiting for pipeline to complete...")
print("(This may take 10-15 minutes)\n")
execution.wait()

# Get execution details
execution_desc = execution.describe()
status = execution_desc['PipelineExecutionStatus']

print(f"\n✅ Pipeline execution complete!")
print(f"   Status: {status}")

✅ Pipeline execution started!
   Execution ARN: arn:aws:sagemaker:us-east-1:513691803389:pipeline/SoundEmotionClassificationPipeline/execution/1lg4yrnr8ues
   Accuracy Threshold: 0.95 (will likely FAIL)
Waiting for pipeline to complete...
(This may take 10-15 minutes)


✅ Pipeline execution complete!
   Status: Succeeded


## 10. View The Results

In [17]:
#10.1 Confirm pipeline ran, and check if threshold was met
steps = execution.list_steps()
print("=" * 80)
print("PIPELINE STEPS - DETAILED STATUS")
print("=" * 80)

for step in steps:
    step_name = step['StepName']
    step_status = step['StepStatus']
    
    # Icon based on status
    icon = "✅" if step_status == "Succeeded" else "❌" if step_status == "Failed" else "⏳"
    
    print(f"\n{icon} Step: {step_name}")
    print(f"   Status: {step_status}")
    
    # Check for condition step
    if "CheckAccuracyThreshold" in step_name:
        try:
            outcome = step['Metadata']['Condition']['Outcome']
            
            # outcome == True means condition PASSED
            condition_passed = (outcome == True or outcome == "true" or outcome == "True")
            
            print(f"\n   🎯 CONDITION EVALUATION:")
            print(f"   Outcome: {outcome}")
            
            # FIXED LOGIC
            if condition_passed:
                print(f"   Result: PASSED ✅ - Model WILL be registered")
            else:
                print(f"   Result: FAILED ❌ - Model will NOT be registered")
            
            # Get threshold value
            try:
                exec_desc = execution.describe()
                params_list = exec_desc.get("PipelineParameters", [])
                params = {p["Name"]: p["Value"] for p in params_list}
                
                if "AccuracyThreshold" in params:
                    threshold_value = float(params["AccuracyThreshold"])
                else:
                    threshold_value = 0.95  # HARD CODED
                
                print(f"   📏 Threshold: {threshold_value:.4f} ({threshold_value * 100:.1f}%)")
                
            except Exception as e:
                print(f"   ⚠️  Could not get threshold: {e}")
                threshold_value = 0.95 #HARD CODED 
            
            # Get actual accuracy value
            try:
                eval_step = next((s for s in steps if s['StepName'] == 'EvaluateModel'), None)
                if eval_step and eval_step['StepStatus'] == 'Succeeded':
                    job_arn = eval_step['Metadata']['ProcessingJob']['Arn']
                    job_name = job_arn.split('/')[-1]
                    job_details = sm_client.describe_processing_job(ProcessingJobName=job_name)
                    
                    s3_uri = job_details['ProcessingOutputConfig']['Outputs'][0]['S3Output']['S3Uri']
                    eval_json_uri = f"{s3_uri}/evaluation.json"
                    
                    bucket_name = eval_json_uri.split('/')[2]
                    key = '/'.join(eval_json_uri.split('/')[3:])
                    s3_client.download_file(bucket_name, key, '/tmp/evaluation.json')
                    
                    with open('/tmp/evaluation.json', 'r') as f:
                        eval_data = json.load(f)
                    
                    accuracy = eval_data['multiclass_classification_metrics']['accuracy']['value']
                    print(f"   📊 Model Accuracy: {accuracy:.4f} ({accuracy * 100:.2f}%)")
                    
                    # FIXED: Show correct comparison
                    if condition_passed:
                        print(f"   ✓ {accuracy * 100:.2f}% ≥ {threshold_value * 100:.1f}% → Model Registered ✅")
                    else:
                        print(f"   ✗ {accuracy * 100:.2f}% < {threshold_value * 100:.1f}% → Model NOT Registered ❌")
                    
            except Exception as e:
                print(f"   ⚠️  Could not retrieve accuracy: {e}")
                
        except KeyError:
            print("   No condition metadata available")

print("\n" + "=" * 80)
print("SUMMARY")
print("=" * 80)

# Check if RegisterModel step ran
register_step = next((s for s in steps if 'Register' in s['StepName']), None)

if register_step:
    print(f"✅ RegisterSoundEmotionModel step: {register_step['StepStatus']}")
    print(f"   Model WAS registered to the registry")
    print(f"   Reason: Accuracy ({accuracy * 100:.2f}%) met threshold ({threshold_value * 100:.1f}%)")
else:
    print(f"⏭️  RegisterSoundEmotionModel step: SKIPPED")
    print(f"   Model was NOT registered (accuracy below threshold)")

print("=" * 80)

PIPELINE STEPS - DETAILED STATUS

✅ Step: CheckAccuracyThreshold
   Status: Succeeded

   🎯 CONDITION EVALUATION:
   Outcome: False
   Result: FAILED ❌ - Model will NOT be registered
   📏 Threshold: 0.9500 (95.0%)
   📊 Model Accuracy: 0.6081 (60.81%)
   ✗ 60.81% < 95.0% → Model NOT Registered ❌

✅ Step: EvaluateModel
   Status: Succeeded

✅ Step: TrainXGBoostModel
   Status: Succeeded

✅ Step: PreprocessSoundData
   Status: Succeeded

SUMMARY
⏭️  RegisterSoundEmotionModel step: SKIPPED
   Model was NOT registered (accuracy below threshold)


## CleanUp Resources

In [ ]:
# Guard cell placed before cleanup
SHOULD_CLEANUP = False  # flip this when you actually want cleanup to run

if not SHOULD_CLEANUP:
    raise RuntimeError("Stopping notebook before cleanup cell.")

In [20]:
from botocore.exceptions import ClientError

# CONFIGURATION
pipeline_name = "SoundEmotionClassificationPipeline"
model_package_group_name = "SoundEmotionModelGroup"
bucket = "sagemaker-us-east-1-418418308994"
prefix = "models/benchmarks"

# Clients
sagemaker = boto3.client('sagemaker')
s3_client = boto3.client('s3')

print("="*70)
print("SAGEMAKER CI/CD PIPELINE CLEANUP")
print("="*70)
print(f"⚠️  WARNING: This will delete pipeline resources")
print(f"Pipeline: {pipeline_name}")
print(f"Model Group: {model_package_group_name}")
print("="*70)

# Ask for confirmation
confirm = input("\nType 'DELETE' to confirm cleanup (or anything else to cancel): ")
if confirm != "DELETE":
    print("\n❌ Cleanup cancelled")
    exit(0)

print("\n✅ Confirmation received. Starting cleanup...")


# STEP 1: DELETE PIPELINE

print("\n" + "="*70)
print("Deleting Pipeline")
print("="*70)

try:
    sagemaker.delete_pipeline(PipelineName=pipeline_name)
    print(f"✅ Deleted pipeline: {pipeline_name}")
except ClientError as e:
    if e.response['Error']['Code'] == 'ResourceNotFound':
        print(f"⚠️  Pipeline not found (already deleted): {pipeline_name}")
    else:
        print(f"❌ Error: {e}")


# STEP 2: DELETE MODEL PACKAGES

print("\n" + "="*70)
print("Deleting Model Packages")
print("="*70)

delete_models = input("\nDelete model registry entries? (yes/no): ").lower()

if delete_models == "yes":
    try:
        # List all model packages in the group
        model_packages = sagemaker.list_model_packages(
            ModelPackageGroupName=model_package_group_name,
            MaxResults=100
        )
        
        if model_packages['ModelPackageSummaryList']:
            print(f"Found {len(model_packages['ModelPackageSummaryList'])} model packages")
            
            # Delete each model package
            for pkg in model_packages['ModelPackageSummaryList']:
                pkg_arn = pkg['ModelPackageArn']
                try:
                    sagemaker.delete_model_package(ModelPackageName=pkg_arn)
                    print(f"✅ Deleted: {pkg_arn.split('/')[-1]}")
                except Exception as e:
                    print(f"❌ Error deleting {pkg_arn}: {e}")
            
            # Wait a bit for deletions to propagate
            print("\n⏳ Waiting 10 seconds for deletions to propagate...")
            time.sleep(10)
            
            # Delete the model package group
            try:
                sagemaker.delete_model_package_group(
                    ModelPackageGroupName=model_package_group_name
                )
                print(f"✅ Deleted model package group: {model_package_group_name}")
            except Exception as e:
                print(f"❌ Error deleting group: {e}")
        else:
            print("⚠️  No model packages found")
            
    except ClientError as e:
        if e.response['Error']['Code'] == 'ResourceNotFound':
            print(f"⚠️  Model package group not found: {model_package_group_name}")
        else:
            print(f"❌ Error: {e}")
else:
    print("⚠️  Skipping model registry cleanup")


# STEP 3: STOP ACTIVE PROCESSING/TRAINING JOBS

print("\n" + "="*70)
print("Stopping Active Jobs")
print("="*70)

# Stop processing jobs
try:
    processing_jobs = sagemaker.list_processing_jobs(
        StatusEquals='InProgress',
        NameContains='sound-emotion'
    )
    
    if processing_jobs['ProcessingJobSummaries']:
        for job in processing_jobs['ProcessingJobSummaries']:
            try:
                sagemaker.stop_processing_job(ProcessingJobName=job['ProcessingJobName'])
                print(f"✅ Stopped processing job: {job['ProcessingJobName']}")
            except Exception as e:
                print(f"⚠️  Error: {e}")
    else:
        print("⚠️  No active processing jobs")
except Exception as e:
    print(f"⚠️  Error checking processing jobs: {e}")

# Stop training jobs
try:
    training_jobs = sagemaker.list_training_jobs(
        StatusEquals='InProgress',
        NameContains='sound-emotion'
    )
    
    if training_jobs['TrainingJobSummaries']:
        for job in training_jobs['TrainingJobSummaries']:
            try:
                sagemaker.stop_training_job(TrainingJobName=job['TrainingJobName'])
                print(f"✅ Stopped training job: {job['TrainingJobName']}")
            except Exception as e:
                print(f"⚠️  Error: {e}")
    else:
        print("⚠️  No active training jobs")
except Exception as e:
    print(f"⚠️  Error checking training jobs: {e}")


# STEP 5: DELETE LOCAL SCRIPT FILES

print("\n" + "="*70)
print("Deleting Local Script Files")
print("="*70)

import os

local_files = [
    'preprocessing.py',
    'train.py',
    'evaluation.py'
]

for fname in local_files:
    if os.path.exists(fname):
        try:
            os.remove(fname)
            print(f"✅ Deleted: {fname}")
        except Exception as e:
            print(f"❌ Error deleting {fname}: {e}")
    else:
        print(f"⚠️  Not found: {fname}")


SAGEMAKER CI/CD PIPELINE CLEANUP
⚠️  WARNING: This will delete pipeline resources
Pipeline: SoundEmotionClassificationPipeline
Model Group: SoundEmotionModelGroup



Type 'DELETE' to confirm cleanup (or anything else to cancel):  DELETE



✅ Confirmation received. Starting cleanup...

Deleting Pipeline
⚠️  Pipeline not found (already deleted): SoundEmotionClassificationPipeline

Deleting Model Packages



Delete model registry entries? (yes/no):  yes


⚠️  No model packages found

Stopping Active Jobs
⚠️  No active processing jobs
⚠️  No active training jobs

Deleting Local Script Files
✅ Deleted: preprocessing.py
✅ Deleted: train.py
✅ Deleted: evaluation.py
